Assignment || 


In [65]:
# data and toolkit 
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd


#datasets from scikit-learn 
from sklearn.datasets import load_diabetes
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris

## for the off-shell / scikit-learn models
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split



In [66]:
diabetes= load_diabetes(scaled=False)  #raw features unscaled
#now extract the features and target 
X=diabetes.data
Y=diabetes.target
# now will be using vectorization do check the shape always
print("featuer shape (X):",X.shape)
print("target shape (X):",Y.shape)

featuer shape (X): (442, 10)
target shape (X): (442,)


In [67]:
#Split protocol (60 / 20 / 20)
X_tr_val, X_te, y_tr_val, y_te = train_test_split(X, Y, test_size=0.20, random_state=42)

X_tr, X_va, y_tr, y_va = train_test_split(X_tr_val, y_tr_val, test_size=0.25, random_state=42)

# Print the sizes to verify they match assignment's sanity table
print(f"Train size: {X_tr.shape[0]}")  
print(f"Validation size: {X_va.shape[0]}")  
print(f"Test size: {X_te.shape[0]}")  
print(f"Total size: {X.shape[0]}")

Train size: 264
Validation size: 89
Test size: 89
Total size: 442


+general idea+
Standardizer in machine learning is the process of rescaling features so that they all share a mean of 0 and a standered deviation of 1 
so for every feature 
z= x-mean / deviation 
so each feature contributes equally to the gradient allowing the learning rate to work properly without blowing up 

In [68]:
#now the standlizer 
class Standardizer:
    def fit(self,X):
        #compute the mean and variance alog the axis 0 --- columns
        self.mu=np.mean(X,axis=0)
        self.sigma=np.std(X,axis=0)
        #handle the assignment restriction on std 0-----> 1 when zero variance column
        self.sigma[self.sigma==0]=1.0
    def transform(self, X):
        return (X-self.mu)/self.sigma
    def fit_transfrom(self,X):
        self.fit(X)
        return self.fit_transfrom(X)

scaler =Standardizer()
scaler.fit(X_tr)  # Fit ONLY on training data!
X_tr_scaled = scaler.transform(X_tr)
X_va_scaled = scaler.transform(X_va)
X_te_scaled = scaler.transform(X_te)
def add_bias(X):
# Adds a column of ones to the left of X -> shape becomes (N, D+1)
    return np.hstack([np.ones((X.shape[0], 1)), X])

# Ready-to-use matrices with bias
X_tr_tilde = add_bias(X_tr_scaled)

why must μ and σ come from the training rows only, and how would computing them on all rows
make your test result misleading?
-there will be data leakage is we compute m , std from all the dataset so the model takes unfair peek at the test distribuation and that will lead to a misleading model 


In [69]:
#regression matrics 
def mse(y_true,y_pred):
    return np.mean((y_true-y_pred)**2)

def rmse(y_true,y_pred):
    return np.sqrt(mse(y_true,y_pred))

def r2(y_true,y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

In [70]:
#classification - cunfusion matrics
def accuracy(y_true,y_pred):
    return np.mean(y_true==y_pred)

def confusion_matrix(y_true,y_pred,K):   # k number of classes / how many categories we're trying to predict
    cm=np.zeros((K,K),dtype=int)
    for t , p in zip(y_true,y_pred):      # zip takes t , p and pair them together as i ++
        cm[t,p]+=1
    return cm

def classification_matrics(cm):
    k=cm.shape[0]
    precisions = []
    recalls=[]
    f1s=[]

    for i in range(K):
        tp=cm[i,i]
        fp=np.sum(cm[:,i])-tp # Column sum minus true positives
        fn=np.sum(cm[i,:])-tp # Row sum minus true positives

        precision = tp/(tp+fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (
            (2 * precision * recall) / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
        
  # Macro averages
    return (
        np.mean(precisions),
        np.mean(recalls),
        np.mean(f1s),
        precisions,
        recalls,
        f1s,
    )

        



In [71]:
#used Ai to test 
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix as sk_confusion_matrix
from sklearn.metrics import precision_recall_fscore_support

# 1. Generate random labels for K = 4 classes and N = 500 samples
rng = np.random.default_rng(42)
K = 4
N = 500
y_true = rng.integers(0, K, size=N)
y_pred = rng.integers(0, K, size=N)

#-- VERIFY CONFUSION MATRIX ---
my_cm = confusion_matrix(y_true, y_pred, K)
sk_cm = sk_confusion_matrix(y_true, y_pred, labels=np.arange(K))

assert np.allclose(
    my_cm, sk_cm, atol=1e-12
), "Confusion matrix does not match sklearn!"
print("✅ Confusion Matrix matches sklearn to 1e-12")

# --- VERIFY ACCURACY ---
my_acc = accuracy(y_true, y_pred)
sk_acc = accuracy_score(y_true, y_pred)

assert np.isclose(
    my_acc, sk_acc, atol=1e-12
), "Accuracy does not match sklearn!"
print("✅ Accuracy matches sklearn to 1e-12")

# --- VERIFY PRECISION, RECALL, F1 (MACRO) ---
# Note: sklearn's precision_recall_fscore_support handles zero division via zero_division=0
my_macro_p, my_macro_r, my_macro_f1, _, _, _ = classification_matrics(my_cm)
sk_macro_p, sk_macro_r, sk_macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)

assert np.isclose(
    my_macro_p, sk_macro_p, atol=1e-12
), "Macro Precision mismatch!"
assert np.isclose(my_macro_r, sk_macro_r, atol=1e-12), "Macro Recall mismatch!"
assert np.isclose(my_macro_f1, sk_macro_f1, atol=1e-12), "Macro F1 mismatch!"
print("✅ Macro Precision, Recall, and F1 match sklearn to 1e-12!")
#end of Ai test

✅ Confusion Matrix matches sklearn to 1e-12
✅ Accuracy matches sklearn to 1e-12
✅ Macro Precision, Recall, and F1 match sklearn to 1e-12!


In [72]:
#helper central difference gradient checker / sanity check 
#Gradient checking solves this by comparing your analytic gradient against a foolproof, brute-force numerical estimate. If they match, your calculus formula is correct!
def grad_check(f,grad_fn , w, eps=1e-5):
    """Checks if the analytic gradient matches the numerical gradient.

  Parameters:
  - f: function that takes w and returns the scalar loss J(w)
  - grad_fn: function that takes w and returns the analytic gradient vector
  g_analytic
  - w: the weight vector to test
  - eps: perturbation step size (default 1e-5 per assignment spec)
  """
    g_analytic = grad_fn(w)
    g_numeric= np.zeros_like(w)  # ---->same shape 

    for i in range(w.size):
        w_origin= w[i]
        #forward steps : w + eps
        w[i]= w_origin + eps
        loss_high= f(w)
        #backword steps : w - eps
        w[i]= w_origin - eps
        loss_low= f(w)

        # Central difference formula: (f(w+eps) - f(w-eps)) / (2*eps)
        g_numeric[i] = (loss_high - loss_low) / (2 * eps)
        #reset w to its original value 
        w[i] = w_origin


        #relative error
        numerator = np.linalg.norm(g_analytic - g_numeric)
        denominator = max(
            np.linalg.norm(g_analytic), np.linalg.norm(g_numeric), 1e-12
        )
        rel_error = numerator / denominator

        print(f"Gradient Check Relative Error: {rel_error:.2e}")

        # Assert it is below the required threshold (1e-6)
        assert (
            rel_error < 1e-6
        ), f"Gradient check FAILED! Relative error is {rel_error}"
        print("Gradient check PASSED!")

        return rel_error



In [ ]:
#logistic reg 